# D_S2 — Deterministic Metrics Computation

Computes all deterministic metrics (no permutation) on the **D experiment** dataset:
- Single unified dataset from `output/S1/` (54,270 cases)
- 27 families (26 signal F01–F26 + Null) × 201 SNR × 10 reps

Metrics include: Pearson, Spearman, distance correlation/covariance,
MINE (MIC/MAS/MEV/MCN), LOWESS, GAM, bin-based, slope-based, distribution, etc.

Output: `output/S2/metrics_full.parquet`

In [1]:
from __future__ import annotations

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, ks_2samp, wasserstein_distance
from statsmodels.nonparametric.smoothers_lowess import lowess

try:
    from minepy import MINE
    HAS_MINEPY = True
except ImportError:
    HAS_MINEPY = False
    print('minepy not installed — MIC/MAS/MEV/MCN will be NaN.')

try:
    from pygam import LinearGAM, s as gam_s
    HAS_PYGAM = True
except ImportError:
    HAS_PYGAM = False
    print('pygam not installed — GAM metrics will be NaN.')

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings('ignore', category=np.RankWarning)

S1_DIR = Path('output/S1')
OUT_DIR = Path('output/S2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [2]:
cases_df = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
pts = np.load(S1_DIR / 'scatter_points.npz')
x_all = pts['x']
y_all = pts['y']
n_cases = len(cases_df)

print(f'Loaded: {n_cases:,} cases × {x_all.shape[1]} points')
print(f'\nCategory breakdown:')
print(cases_df['category'].value_counts().to_string())

del pts

Loaded: 217,080 cases × 500 points

Category breakdown:


KeyError: 'category'

## Metric Functions

In [3]:
def vectorised_pearson(x, y):
    xc = x - x.mean(axis=1, keepdims=True)
    yc = y - y.mean(axis=1, keepdims=True)
    num = (xc * yc).sum(axis=1)
    den = np.sqrt((xc**2).sum(axis=1) * (yc**2).sum(axis=1))
    return np.where(den > 0, num / den, np.nan)


def _rank_rows(arr):
    n_cases, n_points = arr.shape
    ranks = np.empty(arr.shape, dtype=np.float64)
    order = arr.argsort(axis=1)
    rows = np.arange(n_cases)[:, None]
    ranks[rows, order] = np.arange(1, n_points + 1, dtype=np.float64)
    return ranks


def vectorised_spearman(x, y):
    return vectorised_pearson(_rank_rows(x), _rank_rows(y))

In [4]:
def _to_valid(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    return x[valid], y[valid]

def _safe_div(a, b):
    if b == 0 or not np.isfinite(b):
        return np.nan
    return a / b

def _minmax01(v):
    v = np.asarray(v, dtype=float)
    lo, hi = np.nanmin(v), np.nanmax(v)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return np.full_like(v, np.nan, dtype=float)
    return (v - lo) / (hi - lo)

def _endpoint_slope(x, y):
    if len(x) < 2: return np.nan
    return _safe_div(float(y[-1] - y[0]), float(x[-1] - x[0]))

def _polyfit_slope(x, y):
    if len(x) < 3 or np.std(x) == 0: return np.nan
    return float(np.polyfit(x, y, 1)[0])

def _segment_masks(n):
    i1, i2 = n // 3, 2 * n // 3
    early  = np.zeros(n, bool); early[:i1]   = True
    middle = np.zeros(n, bool); middle[i1:i2] = True
    late   = np.zeros(n, bool); late[i2:]    = True
    return early, middle, late

def _residual_sd(y, y_hat):
    r = y - y_hat
    return float(np.nanstd(r, ddof=1)) if len(r) >= 2 else np.nan

def _r2(y, y_hat):
    ss_res = float(np.nansum((y - y_hat)**2))
    ss_tot = float(np.nansum((y - np.nanmean(y))**2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

def _sign_changes(x_curve, y_curve, tol=1e-6):
    x_curve = np.asarray(x_curve, float)
    y_curve = np.asarray(y_curve, float)
    valid = np.isfinite(x_curve) & np.isfinite(y_curve)
    x_curve, y_curve = x_curve[valid], y_curve[valid]
    if len(x_curve) < 4: return np.nan
    order = np.argsort(x_curve)
    x_curve, y_curve = x_curve[order], y_curve[order]
    dx = np.diff(x_curve)
    dy = np.diff(y_curve)
    ok = dx != 0
    if ok.sum() < 3: return np.nan
    slopes = dy[ok] / dx[ok]
    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > tol] = 1
    signs[slopes < -tol] = -1
    nz = signs[signs != 0]
    return float(np.sum(nz[1:] != nz[:-1])) if len(nz) >= 2 else 0.0

def _make_bins(x, n_bins=10, bin_type='equal_width'):
    if bin_type == 'equal_width':
        return np.asarray(pd.cut(x, bins=n_bins, labels=False, include_lowest=True, duplicates='drop'), dtype=float)
    elif bin_type == 'equal_count':
        return np.asarray(pd.qcut(x, q=n_bins, labels=False, duplicates='drop'), dtype=float)
    raise ValueError(f'Unknown bin_type: {bin_type}')

In [5]:
def _double_center(a):
    a = a.reshape(-1, 1)
    dist = squareform(pdist(a))
    return dist - dist.mean(axis=0, keepdims=True) - dist.mean(axis=1, keepdims=True) + dist.mean()

def _distance_metrics(x, y):
    r = {'distance_covariance': np.nan, 'distance_correlation': np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return r
    A, B = _double_center(x), _double_center(y)
    dcov_xy = float(np.sqrt(max((A * B).mean(), 0)))
    dcov_xx = float(np.sqrt(max((A * A).mean(), 0)))
    dcov_yy = float(np.sqrt(max((B * B).mean(), 0)))
    r['distance_covariance'] = dcov_xy
    r['distance_correlation'] = _safe_div(dcov_xy, np.sqrt(dcov_xx * dcov_yy))
    return r

def _mine_metrics(x, y):
    empty = {'MIC': np.nan, 'MAS': np.nan, 'MEV': np.nan, 'MCN': np.nan, 'MIC_minus_r2': np.nan}
    if not HAS_MINEPY or len(x) < 5: return empty
    try:
        mine = MINE(alpha=0.6, c=15)
        mine.compute_score(x, y)
        mic = mine.mic()
        r = pearsonr(x, y)[0] if np.std(x) > 0 and np.std(y) > 0 else np.nan
        return {'MIC': mic, 'MAS': mine.mas(), 'MEV': mine.mev(), 'MCN': mine.mcn(),
                'MIC_minus_r2': mic - r**2 if np.isfinite(r) else np.nan}
    except Exception:
        return empty

def _slope_metrics(x, y, prefix='raw'):
    keys = []
    for method in ['endpoint', 'polyfit']:
        for seg in ['overall', 'early', 'middle', 'late']:
            keys.append(f'{prefix}_{method}_{seg}_slope')
    keys.append(f'{prefix}_segment_strength')
    empty = {k: np.nan for k in keys}
    if len(x) < 6: return empty
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    em, mm, lm = _segment_masks(len(xs))
    segments = {'overall': (xs, ys), 'early': (xs[em], ys[em]),
                'middle': (xs[mm], ys[mm]), 'late': (xs[lm], ys[lm])}
    r = {}
    ep_seg_abs = []
    for seg_name, (sx, sy) in segments.items():
        ep = _endpoint_slope(sx, sy)
        pf = _polyfit_slope(sx, sy)
        r[f'{prefix}_endpoint_{seg_name}_slope'] = ep
        r[f'{prefix}_polyfit_{seg_name}_slope'] = pf
        if seg_name != 'overall':
            ep_seg_abs.append(abs(ep) if np.isfinite(ep) else np.nan)
    r[f'{prefix}_segment_strength'] = float(np.nanmean(ep_seg_abs))
    return r

def _standardized_slope_metrics(x, y):
    xn, yn = _minmax01(x), _minmax01(y)
    if np.any(np.isnan(xn)) or np.any(np.isnan(yn)):
        keys = []
        for method in ['endpoint', 'polyfit']:
            for seg in ['overall', 'early', 'middle', 'late']:
                keys.append(f'standardized_{method}_{seg}_slope')
        keys.append('standardized_segment_strength')
        return {k: np.nan for k in keys}
    return _slope_metrics(xn, yn, prefix='standardized')

def _bin_metrics(x, y, n_bins=10, bin_type='equal_width', min_count=5):
    prefix = f'{bin_type}_bin'
    r = {f'{prefix}_amplitude': np.nan, f'{prefix}_eta_squared': np.nan,
         f'{prefix}_buffer_width_mean': np.nan,
         f'{prefix}_early_buffer_width': np.nan, f'{prefix}_middle_buffer_width': np.nan,
         f'{prefix}_late_buffer_width': np.nan, f'{prefix}_n_valid_bins': 0}
    if len(x) < n_bins: return r
    try: bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception: return r
    df = pd.DataFrame({'x': x, 'y': y, 'bin': bins}).dropna()
    if df.empty: return r
    bin_stats = []
    for bid, g in df.groupby('bin', observed=True):
        if len(g) < min_count: continue
        yv = g['y'].values
        bw = float(np.nanpercentile(yv, 95) - np.nanpercentile(yv, 5))
        bin_stats.append({'bin': bid, 'x_mean': float(g['x'].mean()),
                          'y_mean': float(yv.mean()), 'buffer_width': bw, 'count': len(g)})
    if len(bin_stats) < 2: return r
    bdf = pd.DataFrame(bin_stats).sort_values('x_mean').reset_index(drop=True)
    r[f'{prefix}_amplitude'] = float(bdf['y_mean'].max() - bdf['y_mean'].min())
    y_global = float(df['y'].mean())
    ss_tot = float(np.sum((df['y'].values - y_global)**2))
    ss_bet = sum(row['count'] * (row['y_mean'] - y_global)**2 for _, row in bdf.iterrows())
    r[f'{prefix}_eta_squared'] = _safe_div(ss_bet, ss_tot)
    r[f'{prefix}_buffer_width_mean'] = float(np.nanmean(bdf['buffer_width']))
    r[f'{prefix}_n_valid_bins'] = len(bdf)
    nv = len(bdf)
    i1, i2 = nv // 3, 2 * nv // 3
    r[f'{prefix}_early_buffer_width'] = float(np.nanmean(bdf.iloc[:i1]['buffer_width']))
    r[f'{prefix}_middle_buffer_width'] = float(np.nanmean(bdf.iloc[i1:i2]['buffer_width']))
    r[f'{prefix}_late_buffer_width'] = float(np.nanmean(bdf.iloc[i2:]['buffer_width']))
    return r

def _x_coverage_metrics(x, n_bins=10):
    r = {'x_bin_count_cv': np.nan, 'x_uniform_ks_distance': np.nan}
    xf = x[np.isfinite(x)]
    if len(xf) < 3: return r
    lo, hi = xf.min(), xf.max()
    if hi > lo:
        xn = np.sort((xf - lo) / (hi - lo))
        n = len(xn)
        r['x_uniform_ks_distance'] = float(max(
            np.max(np.arange(1, n+1) / n - xn), np.max(xn - np.arange(0, n) / n)))
    if len(xf) >= n_bins:
        counts, _ = np.histogram(xf, bins=n_bins)
        mu = counts.mean()
        if mu > 0: r['x_bin_count_cv'] = float(np.std(counts, ddof=1) / mu)
    return r

def _lowess_metrics(x, y, frac=0.25):
    empty = {'lowess_residual_sd': np.nan, 'lowess_curve_amplitude': np.nan,
             'lowess_r2': np.nan, 'lowess_first_derivative_sign_changes': np.nan,
             'lowess_overall_slope': np.nan}
    if len(x) < 5: return empty
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    fitted = lowess(ys, xs, frac=frac, return_sorted=True)
    x_fit, y_fit = fitted[:, 0], fitted[:, 1]
    y_pred = np.interp(xs, x_fit, y_fit)
    return {'lowess_residual_sd': _residual_sd(ys, y_pred),
            'lowess_curve_amplitude': float(np.nanmax(y_fit) - np.nanmin(y_fit)),
            'lowess_r2': _r2(ys, y_pred),
            'lowess_first_derivative_sign_changes': _sign_changes(x_fit, y_fit),
            'lowess_overall_slope': _endpoint_slope(x_fit, y_fit)}

def _gam_metrics(x, y, n_splines=10, lam=0.6, grid_size=200):
    empty = {'gam_residual_sd': np.nan, 'gam_curve_amplitude': np.nan,
             'gam_r2': np.nan, 'gam_first_derivative_sign_changes': np.nan,
             'gam_overall_slope': np.nan}
    if not HAS_PYGAM or len(x) < 10: return empty
    try:
        gam = LinearGAM(gam_s(0, n_splines=n_splines), lam=lam).fit(x.reshape(-1, 1), y)
        x_curve = np.linspace(x.min(), x.max(), grid_size)
        y_curve = gam.predict(x_curve.reshape(-1, 1))
        y_pred = np.interp(x, x_curve, y_curve)
        return {'gam_residual_sd': _residual_sd(y, y_pred),
                'gam_curve_amplitude': float(np.nanmax(y_curve) - np.nanmin(y_curve)),
                'gam_r2': _r2(y, y_pred),
                'gam_first_derivative_sign_changes': _sign_changes(x_curve, y_curve),
                'gam_overall_slope': _endpoint_slope(x_curve, y_curve)}
    except Exception:
        return empty

def _correlation_extra(x, y):
    r = {'covariance': np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return r
    r['covariance'] = float(np.cov(x, y, ddof=1)[0, 1])
    return r

def _distribution_metrics(x, y, n_bins=10, bin_type='equal_width'):
    prefix = f'{bin_type}_distribution'
    r = {f'{prefix}_ks_distance': np.nan, f'{prefix}_wasserstein_distance': np.nan}
    if len(x) < n_bins: return r
    try: bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception: return r
    df = pd.DataFrame({'x': x, 'y': y, 'bin': bins}).dropna()
    if df.empty: return r
    valid_bins = np.sort(df['bin'].unique())
    if len(valid_bins) < 3: return r
    nv = len(valid_bins)
    y_low = df[df['bin'].isin(valid_bins[:nv//3])]['y'].values
    y_high = df[df['bin'].isin(valid_bins[2*nv//3:])]['y'].values
    if len(y_low) < 2 or len(y_high) < 2: return r
    r[f'{prefix}_ks_distance'] = float(ks_2samp(y_low, y_high).statistic)
    r[f'{prefix}_wasserstein_distance'] = float(wasserstein_distance(y_low, y_high))
    return r

_Y_SCALE_METRICS = [
    'equal_width_distribution_wasserstein_distance', 'equal_count_distribution_wasserstein_distance',
    'equal_width_bin_amplitude', 'equal_width_bin_buffer_width_mean',
    'equal_width_bin_early_buffer_width', 'equal_width_bin_middle_buffer_width',
    'equal_width_bin_late_buffer_width',
    'equal_count_bin_amplitude', 'equal_count_bin_buffer_width_mean',
    'equal_count_bin_early_buffer_width', 'equal_count_bin_middle_buffer_width',
    'equal_count_bin_late_buffer_width',
    'lowess_residual_sd', 'lowess_curve_amplitude',
    'gam_residual_sd', 'gam_curve_amplitude',
]

def _ysd_normalized(y, metrics):
    y_sd = float(np.nanstd(y, ddof=1))
    r = {'y_sd': y_sd}
    for m in _Y_SCALE_METRICS:
        if m in metrics:
            r[f'{m}_div_y_sd'] = _safe_div(metrics[m], y_sd)
    return r

In [6]:
def compute_all_per_case(x, y):
    x, y = _to_valid(x, y)
    m = {}
    m.update(_correlation_extra(x, y))
    m.update(_distance_metrics(x, y))
    m.update(_x_coverage_metrics(x))
    m.update(_distribution_metrics(x, y, bin_type='equal_width'))
    m.update(_distribution_metrics(x, y, bin_type='equal_count'))
    m.update(_slope_metrics(x, y, prefix='raw'))
    m.update(_standardized_slope_metrics(x, y))
    m.update(_mine_metrics(x, y))
    m.update(_bin_metrics(x, y, bin_type='equal_width'))
    m.update(_bin_metrics(x, y, bin_type='equal_count'))
    m.update(_lowess_metrics(x, y))
    m.update(_gam_metrics(x, y))
    m.update(_ysd_normalized(y, m))
    m['n_valid'] = len(x)
    return m

# Quick timing test
x_test = x_all[0].astype(np.float64)
y_test = y_all[0].astype(np.float64)
t0 = time.time()
_ = compute_all_per_case(x_test, y_test)
dt = time.time() - t0
print(f'Per-case time: {dt:.2f}s')
print(f'Estimated total: {dt * n_cases / 3600:.1f} hours')

Per-case time: 0.08s
Estimated total: 3.3 hours


## Compute All Metrics

In [7]:
# Phase 1: vectorised Pearson + Spearman
t0 = time.time()
x64 = x_all.astype(np.float64)
y64 = y_all.astype(np.float64)
pearson_all = vectorised_pearson(x64, y64)
spearman_all = vectorised_spearman(x64, y64)
del x64, y64
print(f'Phase 1 done: Pearson + Spearman for {n_cases:,} cases in {time.time()-t0:.1f}s')

/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_244/265413197.py:6: RuntimeWarning: invalid value encountered in divide
  return np.where(den > 0, num / den, np.nan)


Phase 1 done: Pearson + Spearman for 158,790 cases in 10.4s


In [8]:
# Phase 2: per-case metrics
per_case_results = []
t0 = time.time()

for i in tqdm(range(n_cases), desc='Per-case metrics'):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    per_case_results.append(compute_all_per_case(x, y))

    if (i + 1) % 10000 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (n_cases - i - 1) / rate
        print(f'  {i+1:>7,}/{n_cases:,}  ({rate:.0f} cases/s, ETA {eta/60:.1f} min)')

elapsed = time.time() - t0
print(f'Phase 2 done: {n_cases:,} cases in {elapsed/60:.1f} min')

Per-case metrics:   0%|          | 0/158790 [00:00<?, ?it/s]

Per-case metrics:   6%|▋         | 10004/158790 [06:31<1:28:50, 27.91it/s]

   10,000/158,790  (26 cases/s, ETA 97.1 min)


Per-case metrics:  13%|█▎        | 20004/158790 [13:32<1:20:56, 28.58it/s]

   20,000/158,790  (25 cases/s, ETA 93.9 min)


...

did not converge
did not converge
did not converge


/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pygam/pygam.py:806: RuntimeWarning: divide by zero encountered in scalar divide
  diff = np.linalg.norm(self.coef_ - coef_new) / np.linalg.norm(coef_new)
/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pygam/pygam.py:806: RuntimeWarning: invalid value encountered in scalar divide
  diff = np.linalg.norm(self.coef_ - coef_new) / np.linalg.norm(coef_new)
/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/scipy/stats/_distn_infrastructure.py:2029: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pygam/distributions.py:195: RuntimeWarning: invalid value encountered in divide
  dev /= self.scale
/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pygam/pygam.py:1291: RuntimeWarning: invalid value encountered in scalar divide
  score = score / rank
/Users/mimi/miniconda3/envs/pip39/lib/p

did not converge
did not converge
did not converge


/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/scipy/stats/_distn_infrastructure.py:2029: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pygam/distributions.py:195: RuntimeWarning: invalid value encountered in divide
  dev /= self.scale
/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pygam/pygam.py:1291: RuntimeWarning: invalid value encountered in scalar divide
  score = score / rank
/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pygam/pygam.py:806: RuntimeWarning: divide by zero encountered in scalar divide
  diff = np.linalg.norm(self.coef_ - coef_new) / np.linalg.norm(coef_new)
/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/pygam/pygam.py:806: RuntimeWarning: invalid value encountered in scalar divide
  diff = np.linalg.norm(self.coef_ - coef_new) / np.linalg.norm(coef_new)
/Users/mimi/miniconda3/envs/pip39/lib/p

did not converge
Phase 2 done: 158,790 cases in 106.9 min


## Save Results

In [9]:
metrics_df = pd.DataFrame(per_case_results)
metrics_df.insert(0, 'case_id', cases_df['case_id'].values)
metrics_df['pearson_r'] = pearson_all
metrics_df['spearman_rho'] = spearman_all

out_path = OUT_DIR / 'metrics_full.parquet'
metrics_df.to_parquet(out_path, index=False)
print(f'Saved {out_path}  ({len(metrics_df):,} rows × {len(metrics_df.columns)} columns)')
print(f'Columns: {list(metrics_df.columns)[:10]} ... ({len(metrics_df.columns)} total)')

Saved output/S2/metrics_full.parquet  (158,790 rows × 77 columns)
Columns: ['case_id', 'covariance', 'distance_covariance', 'distance_correlation', 'x_bin_count_cv', 'x_uniform_ks_distance', 'equal_width_distribution_ks_distance', 'equal_width_distribution_wasserstein_distance', 'equal_count_distribution_ks_distance', 'equal_count_distribution_wasserstein_distance'] ... (77 total)
